### Setups

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, cohen_kappa_score, precision_score, recall_score
from matplotlib.patches import Patch

Path("../figures/statistical_evaluation").mkdir(exist_ok=True)

### Load data

In [6]:
# Majority
#P_MAJ_ALL = "pcm_models_results/personai/majority_class_parrot_all_user_results.json"
#D_MAJ_ALL = "pcm_models_results/dorn/majority_class_parrot_all_user_results.json" 

# PCM model(found in the notebook generate_confusion_matrix.ipynb, but setting both to random_forest)       
P_PCM_ALL = "../../PCM/results/personai/random_forest_all_user_results.json"       
D_PCM_ALL = "../../PCM/results/dorn/random_forest_all_user_results.json"

#Best Cohort Model Warm
P_BEST_COHORT_ALL = "../../Experiments/PersonAI_warm/PersonAI_warm_Cross_Dist_survey/meta.json"
D_BEST_COHORT_ALL = "../../Experiments/dorn_warm/dorn_warm_Dist_Cross/meta.json"
personai_best_exp = "../../Experiments/PersonAI_warm/PersonAI_warm_Cross_Dist_survey"
dorn_best_exp = "../../Experiments/dorn_warm/dorn_warm_Dist_Cross"

#Best Cohort Model Cold
P_BEST_COHORT_ALL_COLD = "../../Experiments/PersonAI_cold/PersonAI_cold_sex/meta.json"
D_BEST_COHORT_ALL_COLD = "../../Experiments/dorn_cold/dorn_cold_emotional_stability/meta.json"
personai_best_exp_cold = "../../Experiments/PersonAI_cold/PersonAI_cold_sex"
dorn_best_exp_cold = "../../Experiments/dorn_cold/dorn_cold_emotional_stability"


### helper functions

In [3]:
from statistics import median


def load_pcm_like_metrics(all_user_results_json):
    d = json.load(open(all_user_results_json, "r", encoding="utf-8"))
    users = d["all_users_results"]

    f1_micro = []
    f1_macro = []
    kappa = []
    precision_macro = []
    recall_macro = []

    for u in users:
        f1_micro.append(users[u]["average_f1_micro_over_all_folds"])
        f1_macro.append(users[u]["average_f1_macro_over_all_folds"])
        kappa.append(users[u]["average_kappa_quadratic_over_all_folds"])

        fold_precisions = []
        fold_recalls = []

        for y_true, y_pred in zip(users[u]["k_folds_y_test"], users[u]["k_folds_y_pred"]):
            fold_precisions.append(
                precision_score(y_true, y_pred, average="macro", zero_division=0)
            )
            fold_recalls.append(
                recall_score(y_true, y_pred, average="macro", zero_division=0)
            )

        precision_macro.append(np.mean(fold_precisions))
        recall_macro.append(np.mean(fold_recalls))

    return np.array(f1_micro), np.array(f1_macro), np.array(kappa), np.array(precision_macro), np.array(recall_macro)

def cohort_user_metrics(dataset_name, exp_dir, baseline=False, majority=False):
    exp_dir = Path(exp_dir)

    dfs = []
    for iter_dir in exp_dir.iterdir():
        if not iter_dir.is_dir():
            continue

        if baseline:
            csvs = list(iter_dir.glob("*_baseline_predictions.csv"))
        else:
            csvs = list(iter_dir.glob("*_cluster_predictions.csv"))

        if not csvs:
            continue

        df = pd.read_csv(csvs[0])
        df.columns = df.columns.str.strip().str.replace("\ufeff", "")
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)

    f1_micro = []
    f1_macro = []
    kappa_quad = []
    precision_macro = []
    recall_macro = []

    for u, g in df_all.groupby("user"):
        if majority:
            y_true = g["true"].astype(int).to_numpy()
            y_pred = g["majority_pred"].astype(int).to_numpy()
        else:
            y_true = g["true"].astype(int).to_numpy()
            y_pred = g["pred"].astype(int).to_numpy()

        f1_micro.append(f1_score(y_true, y_pred, average="micro"))
        f1_macro.append(f1_score(y_true, y_pred, average="macro"))
        kappa_quad.append(cohen_kappa_score(y_true, y_pred, weights="quadratic"))
        precision_macro.append(precision_score(y_true, y_pred,average="macro", zero_division=0))
        recall_macro.append(recall_score(y_true, y_pred,average="macro", zero_division=0))

    return np.array(f1_micro), np.array(f1_macro), np.array(kappa_quad), np.array(precision_macro), np.array(recall_macro)

def evaluate_experiment(dataset_name, exp_path, baseline=False):
    exp_path = Path(exp_path)

    all_true = []
    all_pred = []
    micro_f1 = []
    macro_f1 = []

    for iter_dir in exp_path.iterdir():
        if not iter_dir.is_dir():
            continue

        if baseline:
            csv_files = list(iter_dir.glob("*_baseline_predictions.csv"))
        else:
            csv_files = list(iter_dir.glob("*_cluster_predictions.csv"))

        if not csv_files:
            continue

        df = pd.read_csv(csv_files[0])
        for user in df["user"].unique():
            user_df = df[df["user"] == user]
            all_true.extend(user_df["true"])
            all_pred.extend(user_df["pred"])

            macro_f1.append(f1_score(all_true, all_pred, average="macro"))
            micro_f1.append(f1_score(all_true, all_pred, average="micro"))



    return median(macro_f1), median(micro_f1)


def top_n_cohort_experiments(dataset_name,root_dir, n=3, ignore_substrings=None, baseline=False):
    root_dir = Path(root_dir)
    ignore_substrings = ignore_substrings or []

    scored = []
    for exp in root_dir.iterdir():
        if not exp.is_dir():
            continue
        if any(s in str(exp) for s in ignore_substrings):
            continue

        macro_f1, micro_f1 = evaluate_experiment(dataset_name, exp, baseline=baseline)
        scored.append((float(micro_f1), exp))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:n]



### find top 3 cohorts

In [7]:
'''top3_dorn = top_n_cohort_experiments("dorn", r"cohort_experiments/dorn", n=3)
top3_personai = top_n_cohort_experiments("personai", r"cohort_experiments/personai", n=3, ignore_substrings=["cold"])

print("Top 3 DORN:")
for macro_f1, micro_f1, p in top3_dorn:
    print(f"{p}: {macro_f1}")

print("Top 3 PersonAI:")
for macro_f1, micro_f1, p in top3_personai:
    print(f"{p}: {macro_f1}")'''


p_macro_f1, p_micro_f1 = evaluate_experiment("personai", personai_best_exp, baseline=False)
print(f"{personai_best_exp}: Macro F1 = {p_macro_f1}, Micro F1 = {p_micro_f1}")
p_macro_f1_b, p_micro_f1_b = evaluate_experiment("personai", personai_best_exp, baseline=True)
print(f"{personai_best_exp} (baseline): Macro F1 = {p_macro_f1_b}, Micro F1 = {p_micro_f1_b}")

d_macro_f1, d_micro_f1 = evaluate_experiment("dorn", dorn_best_exp, baseline=False)
print(f"{dorn_best_exp}: Macro F1 = {d_macro_f1}, Micro F1 = {d_micro_f1}")
d_macro_f1_b, d_micro_f1_b = evaluate_experiment("dorn", dorn_best_exp, baseline=True)
print(f"{dorn_best_exp} (baseline): Macro F1 = {d_macro_f1_b}, Micro F1 = {d_micro_f1_b}")



../../Experiments/PersonAI_warm/PersonAI_warm_Cross_Dist_survey: Macro F1 = 0.35101858109271744, Micro F1 = 0.7226253459035115
../../Experiments/PersonAI_warm/PersonAI_warm_Cross_Dist_survey (baseline): Macro F1 = 0.30879703188356167, Micro F1 = 0.6854945240966951
../../Experiments/dorn_warm/dorn_warm_Dist_Cross: Macro F1 = 0.399125551832139, Micro F1 = 0.654945054945055
../../Experiments/dorn_warm/dorn_warm_Dist_Cross (baseline): Macro F1 = 0.2722129211350841, Micro F1 = 0.4453463203463204


In [8]:
top3_dorn = top_n_cohort_experiments("dorn", r"../../Experiments/dorn_cold", n=10)
top3_personai = top_n_cohort_experiments("PersonAI", r"../../Experiments/PersonAI_cold", n=10)

print("Top 10 DORN:")
for micro_f1, p in top3_dorn:
    print(f"{p}: {micro_f1}")

print("Top 10 PersonAI:")
for micro_f1, p in top3_personai:
    print(f"{p}: {micro_f1}")


#dorn_cold_emotional_stability: 0.569925444925445
#PersonAI_cold_sex: 0.6997790868924889


p_macro_f1, p_micro_f1 = evaluate_experiment("personai", personai_best_exp_cold, baseline=False)
print(f"{personai_best_exp_cold}: Macro F1 = {p_macro_f1}, Micro F1 = {p_micro_f1}")
p_macro_f1_b, p_micro_f1_b = evaluate_experiment("personai", personai_best_exp_cold, baseline=True)
print(f"{personai_best_exp_cold} (baseline): Macro F1 = {p_macro_f1_b}, Micro F1 = {p_micro_f1_b}")

d_macro_f1, d_micro_f1 = evaluate_experiment("dorn", dorn_best_exp_cold, baseline=False)
print(f"{dorn_best_exp_cold}: Macro F1 = {d_macro_f1}, Micro F1 = {d_micro_f1}")
d_macro_f1_b, d_micro_f1_b = evaluate_experiment("dorn", dorn_best_exp_cold, baseline=True)
print(f"{dorn_best_exp_cold} (baseline): Macro F1 = {d_macro_f1_b}, Micro F1 = {d_micro_f1_b}")


Top 10 DORN:
../../Experiments/dorn_cold/dorn_cold_survey: 0.5924963924963925
../../Experiments/dorn_cold/dorn_cold_emotional_stability: 0.5910808767951625
../../Experiments/dorn_cold/dorn_cold_openness_to_experiences_wrong: 0.5748524203069658
../../Experiments/dorn_cold/dorn_cold_swls_wrong: 0.5627705627705628
../../Experiments/dorn_cold/dorn_cold_conscientiousness: 0.560482374768089
../../Experiments/dorn_cold/dorn_cold_agreeableness: 0.5505050505050505
../../Experiments/dorn_cold/dorn_cold_big5_wrong: 0.5492063492063493
../../Experiments/dorn_cold/dorn_cold_sex: 0.5414862914862915
../../Experiments/dorn_cold/dorn_cold_sex_wrong: 0.5402708402708403
../../Experiments/dorn_cold/dorn_cold_big5: 0.5353019995877139
Top 10 PersonAI:
../../Experiments/PersonAI_cold/PersonAI_cold_bfi2s_agreeableness_wrong: 0.7077668340146589
../../Experiments/PersonAI_cold/PersonAI_cold_swls_score_filtered: 0.7064134119564072
../../Experiments/PersonAI_cold/PersonAI_cold_swls_score: 0.7039399012253893
../../

### metrics overview

In [9]:
#D_maj_micro, D_maj_macro, D_maj_kappa, D_maj_pre, D_maj_recall = load_pcm_like_metrics(D_MAJ_ALL)
D_pcm_micro, D_pcm_macro, D_pcm_kappa, D_pcm_pre, D_pcm_recall = load_pcm_like_metrics(D_PCM_ALL)
D_co_micro, D_co_macro, D_co_kappa, D_co_pre, D_co_recall = cohort_user_metrics("dorn", dorn_best_exp)
D_co_micro_b, D_co_macro_b, D_co_kappa_b, D_co_pre_b, D_co_recall_b = cohort_user_metrics("dorn", dorn_best_exp, baseline=True)
D_co_micro_m, D_co_macro_m, D_co_kappa_m, D_co_pre_m, D_co_recall_m = cohort_user_metrics("dorn", dorn_best_exp, majority=True)

D_co_micro_cold, D_co_macro_cold, D_co_kappa_cold, D_co_pre_cold, D_co_recall_cold = cohort_user_metrics("dorn", dorn_best_exp_cold)

print("DORN users (majority):", len(D_co_macro_m))
print("DORN users (pcm):", len(D_pcm_macro))
print("DORN users (cohort):", len(D_co_macro))
print("DORN users (cohort baseline):", len(D_co_macro_b))
print("DORN users (cohort cold):", len(D_co_macro_cold))

#P_maj_micro, P_maj_macro, P_maj_kappa, P_maj_pre, P_maj_recall = load_pcm_like_metrics(P_MAJ_ALL)
P_pcm_micro, P_pcm_macro, P_pcm_kappa, P_pcm_pre, P_pcm_recall = load_pcm_like_metrics(P_PCM_ALL)
P_co_micro, P_co_macro, P_co_kappa, P_co_pre, P_co_recall = cohort_user_metrics("personai", personai_best_exp)
P_co_micro_b, P_co_macro_b, P_co_kappa_b, P_co_pre_b, P_co_recall_b = cohort_user_metrics("personai", personai_best_exp, baseline=True)
P_co_micro_m, P_co_macro_m, P_co_kappa_m, P_co_pre_m, P_co_recall_m = cohort_user_metrics("personai", personai_best_exp, majority=True)

P_co_micro_cold, P_co_macro_cold, P_co_kappa_cold, P_co_pre_cold, P_co_recall_cold = cohort_user_metrics("personai", personai_best_exp_cold)

print("PersonAI users (majority):", len(P_co_macro_m))
print("PersonAI users (pcm):", len(P_pcm_macro))
print("PersonAI users (cohort):", len(P_co_macro))
print("PersonAI users (cohort baseline):", len(P_co_macro_b))
print("PersonAI users (cohort cold):", len(P_co_macro_cold))

DORN users (majority): 20
DORN users (pcm): 20
DORN users (cohort): 20
DORN users (cohort baseline): 20
DORN users (cohort cold): 20
PersonAI users (majority): 72
PersonAI users (pcm): 72
PersonAI users (cohort): 72
PersonAI users (cohort baseline): 72
PersonAI users (cohort cold): 72


### Statistical evaluation

In [12]:
from scipy.stats import ttest_ind, ttest_rel
from numpy import std, mean, sqrt

labels = ["PCM micro","PCM macro","Cohort Warm micro","Cohort Warm macro", "Cohort Cold micro", "Cohort Cold macro", "Cohort Baseline micro", "Cohort Baseline macro", "Majority micro","Majority macro"]

def cohen_d(x,y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (mean(x) - mean(y)) / sqrt(((nx-1)*std(x, ddof=1) ** 2 + (ny-1)*std(y, ddof=1) ** 2) / dof)


def estimate_necessary_sample_size(dataset, data, col_a, col_b):
    df = pandas.DataFrame(np.array(data).T, columns=labels)

    diff = df[col_a] - df[col_b]
    meand = diff.mean()
    stdd = diff.std()

    # for n ≈ (z1−α/2 + z1−β)^2 / δ^2
    # where z is the z score for confidence interval and power, alpha is aimed at 0.95, beta is aimed at 0.8
    # so the corresponding z scores are 1.96 and 0.8416.
    # sigma is the mean of the difference (the thing we want to test, so difference between two model scores) and SD is the standard deviation of that difference
    # δ = mean(d) / SD(d)
    sigm = meand/stdd
    n = (1.96+0.8416)**2 / (sigm**2)
    # n = (2.5+0.8416)**2 / (sigm**2) # for p < 0.00625 bonferroni corrected for 8 comparisons
    print(f'{dataset} {col_a}, {col_b} -> estimated minimum n: {n}')
    return n


def statistical_tests(dataset, data, colx, coly):
    df = pandas.DataFrame(np.array(data).T, columns=labels)

    x,y = df[colx], df[coly]

    stat, pval = ttest_ind(x,y, alternative='greater')
    cohend = cohen_d(x,y)
    print(f'{dataset} {colx + " vs " + coly}: pval: {pval}  cohend: {cohend} mean: {df[colx].mean()}  std: {df[colx].std()}')
    est_n = estimate_necessary_sample_size(dataset, data, colx, coly)
    return {'dataset': dataset, 'colx': colx, 'coly': coly, 'pval': pval, 'cohend': cohend, 'est_n': est_n, 'mean': df[colx].mean(), 'std': df[colx].std()}

stats_res = []
for dataset in ["dorn", "personai"]:
    if dataset == "dorn":    
        data = [D_pcm_micro, D_pcm_macro, D_co_micro, D_co_macro, D_co_micro_cold, D_co_macro_cold, D_co_micro_b, D_co_macro_b, D_co_micro_m, D_co_macro_m]
    else:
        data = [P_pcm_micro, P_pcm_macro, P_co_micro, P_co_macro, P_co_micro_cold, P_co_macro_cold, P_co_micro_b, P_co_macro_b, P_co_micro_m, P_co_macro_m]

    stats_res.append(statistical_tests(dataset, data, "Cohort Baseline macro", "Majority macro"))
    stats_res.append(statistical_tests(dataset, data, "Cohort Cold macro", "Cohort Baseline macro"))
    stats_res.append(statistical_tests(dataset, data, "Cohort Warm macro", "Cohort Baseline macro"))

print(pandas.DataFrame(stats_res).to_markdown())



dorn Cohort Baseline macro vs Majority macro: pval: 0.31804434329288933  cohend: 0.15084303658969675 mean: 0.265742538896076  std: 0.13299546750945676
dorn Cohort Baseline macro, Majority macro -> estimated minimum n: 333.71622186505846
dorn Cohort Cold macro vs Cohort Baseline macro: pval: 0.1481685273340103  cohend: 0.3348487064656691 mean: 0.30442895361373956  std: 0.09491270631489279
dorn Cohort Cold macro, Cohort Baseline macro -> estimated minimum n: 92.0057023961709
dorn Cohort Warm macro vs Cohort Baseline macro: pval: 0.026112733303831313  cohend: 0.6337513211062411 mean: 0.3374945978295448  std: 0.08915625608675924
dorn Cohort Warm macro, Cohort Baseline macro -> estimated minimum n: 42.75356779228483
personai Cohort Baseline macro vs Majority macro: pval: 0.06245861461322253  cohend: 0.25726232738577204 mean: 0.3073997244087412  std: 0.09046286150427602
personai Cohort Baseline macro, Majority macro -> estimated minimum n: 51.4756668376582
personai Cohort Cold macro vs Cohor